# 06 · Feature importance — GPU models (XGBoost, MLP, GNNs) and GNNExplainer

Step 06 explains the models trained in step 04, in three notebooks:
- `06-importance-cpu.ipynb` (CPU): Logistic Regression and Random Forest;
- **this one** (GPU): XGBoost, MLP, GraphSAGE, GATv2, ChebNet and GIN (96 triples), and GNNExplainer on the 20 (menu, GNN) pairs;
- `06-importance-plots.ipynb` (CPU, after the other two): tables and plots.

**Runtime: GPU** (`Runtime → Change runtime type → T4 GPU`). Expected time: a few hours, probably over two Colab sessions (the run resumes where it stopped): Integrated Gradients on the GNNs and GNNExplainer are most of it. A timing cell (section 4) measures the speed first and prints the estimate; if Integrated Gradients on the GNNs would take more than the budget, it runs on seed 0 only.

The menu `none` has no columns: its GNNs get no permutation and no attribution, only the reload check and GNNExplainer on the edges.

**The two questions.** Step 05 compared menus: models **retrained** on different groups of features. This step looks **inside** the trained models of step 04 (no new training) and asks, for every triple (menu, model, seed): which columns and which **families** of columns does the model use, and in which direction?

**Families.** The columns of a menu belong to six families, as decided in step 02: LF (93 Local features of Elliptic), AF (72 Aggregate features), A (6 degrees and counts), B (11 centralities), C (7 motifs and curvature), D (9 economic columns of Elliptic++). A menu with two families or more gets one value per family; `local` (LF only) and `econ` (D only) do not, because shuffling their only family would shuffle the whole menu.

**1 · Permutation importance** (Breiman, 2001; Fisher, Rudin and Dominici, 2019), for every model. The values of one column, or of all the columns of a family together (the same permutation for all of them, so a transaction keeps its own combination), are shuffled among the test transactions; the model is **not** retrained; the importance is the drop of the test PR-AUC:
$$I_j = \mathrm{AP}(\text{test}) - \frac{1}{R}\sum_{r=1}^{R}\mathrm{AP}(\text{test with column } j \text{ shuffled by permutation } r), \qquad R = 5.$$
- The values are shuffled **within each time step**: a transaction of time step 42 never gets a value of time step 47, so the drop measures only "the model uses this column", not "the value comes from another period".
- The 5 permutations come from fixed seeds (0–4) and are **the same for every model** (one set for the evaluation rows, used by LogReg, RF, XGBoost and MLP; one set for all the nodes of the test graph, used by the GNNs, whose neighbours are shuffled too).
- On the validation (time steps 31–40) the same is done **for the families only**, to see whether what a model relies on changes between the two periods.
- The drop of the ROC-AUC is saved too.

**2 · Attributions** $\phi_{ij}$: how much column $j$ pushes transaction $i$ towards "illicit". For every model the same definition (SHAP values, Lundberg and Lee, 2017), computed with the exact method of the model:
- **Logistic Regression** — exact formula, in log-odds: $\phi_{ij} = w_j\,(x_{ij} - \bar x_j)$, with $\bar x$ the mean of the train rows; base value $b + w\cdot\bar x$, and base + $\sum_j \phi_{ij}$ = the logit.
- **Random Forest** — TreeSHAP (Lundberg et al., 2020), exact for trees, in probability; on the explain sample (below), because the forest has deep trees.
- **XGBoost** — TreeSHAP of XGBoost itself (`pred_contribs`), in log-odds, on the whole test.
- **MLP and GNNs** — **Integrated Gradients** (Sundararajan, Taly and Yan, 2017), in logits: with the baseline $x'$ = the mean of the train,
$$\phi_{ij} = (x_{ij} - x'_j)\,\frac{1}{m}\sum_{k=1}^{m} \frac{\partial f}{\partial x_j}\Big(x' + \tfrac{k - 1/2}{m}(x_i - x')\Big), \qquad m = 50,$$
and $\sum_j \phi_{ij} = f(x_i) - f(x')$ (completeness, checked). For a linear model it gives exactly the formula of the Logistic Regression. For a GNN the logit of a node depends also on the features of its neighbours: the gradient is taken with respect to the features of **every** node of its neighbourhood, so each $\phi_{ij}$ splits into the part of the node itself and the part of its neighbours.

Per column and per family (sum of the columns of the family, $\Phi_{iF} = \sum_{j\in F}\phi_{ij}$): the mean of $|\phi|$ over the test, the mean over the illicit transactions, the share of the total, the **density** = share / share of the columns ($> 1$: the family weighs more than its number of columns), the **direction** (Spearman correlation between the value of a column and its $\phi$: $> 0$ means high values push towards "illicit") and, for the GNNs, the **share of the neighbours**.

**The explain sample** (one for every model, in `explain_sample.csv`): all 524 illicit transactions of the test plus 1,000 licit ones drawn in proportion to the time steps (seed 0). Each row has a weight, the number of test transactions it stands for (illicit 1, licit 9,342 / 1,000 = 9.342), so weighted means over the sample estimate means over the test. Inside it, the 80 nodes of the GNNExplainer core (60 illicit in proportion to the time steps, 20 licit).

**3 · GNNExplainer** (Ying et al., 2019), on the 20 pairs with a GNN, seed 0. For one node it learns a soft mask in $[0, 1]$ on every edge of its neighbourhood (and one mask value per column) that keeps the **logit** of the node (the score before the sigmoid) with as few edges and columns as possible: *which neighbours did the GNN look at?* The logit, not the predicted class: on a node predicted with confidence, the loss on the class is saturated (gradient about 0) and the masks would barely move from their random start; the toy test below showed it. So a mask says how much the score of the node **depends** on an edge, not in which direction: an edge that pushes the score down is kept too, if the logit needs it. The nodes: the 80 of the **core** (the same for every GNN), plus 20 licit transactions **flagged by mistake** by that model (false positives, drawn with a fixed seed). Each node is labelled TP, FN, FP or TN at the threshold chosen on validation, and the results are read stratum by stratum. 200 optimisation steps per node. **Stability**: 20 core nodes are explained again with another seed of the explainer, and on the models of seeds 1 and 2.

| File (in `06-feature-importance/`) | Content |
|---|---|
| `explain_sample.csv` | the explain sample: `row`, `txId`, time step, label, weight, order of the licit rows, GNNExplainer core |
| `permutation/<triple>.csv` | one row per column and per family: data (test or validation), mean and standard deviation over the 5 shuffles of the drop of PR-AUC and ROC-AUC |
| `shap_summary/<triple>.csv` | one row per column, per family and for the whole menu: method, mean $|\phi|$, mean $|\phi|$ on the illicit, mean $\phi$, share, density, direction, share of the neighbours (GNNs) |
| `shap/<triple>.parquet` | seed 0 only: the matrix $\phi$ (rows × columns, plus the neighbour part for the GNNs), with `row`, label, weight, base value and output |
| `checks.csv` | every check: reloaded models give the saved scores, additivity / completeness, subgraph = whole graph |
| `done/<triple>.json` | times and settings; written last: its presence means "triple finished" |
| `gnnexplainer/<pair>__*.{parquet,csv}` | GNNExplainer (section 6): edges near each node, mean mask by distance and class of the neighbour, column masks, stability |
| `logs/06-importance-gpu.log` | the log lines |

**Resume.** A triple is skipped when its `done` file exists, a GNNExplainer pair when its `.json` exists. If Colab disconnects, run the notebook again from the top: at most the triple that was running is lost. Files are written through a temporary file and renamed.

**Shared code.** The cells `SHARED CODE` are identical copies of the cells of steps 03 and 04 (data, transformations, models): the models are rebuilt exactly as they were trained. The notebook stops if the files of step 04 were written by another version of that code. The cell `IMPORTANCE CODE` is identical in `06-importance-cpu` and `06-importance-gpu`.

## 0. Setup

In [1]:
# Install the libraries that Colab does not have by default (only if they are missing).
import importlib.util
import subprocess
import sys

for package, module in [('torch_geometric', 'torch_geometric')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

In [2]:
# Connect Google Drive: every input and output of the project lives in DATA_PATH.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DATA_PATH = Path('/content/drive/MyDrive/UNIVERSITA/block-crypto/data')
INPUT_DIR = DATA_PATH / '02-feature-menus'          # output of step 02
RUNS_DIR = DATA_PATH / '04-final-runs'              # output of step 04: models, predictions, summaries
OUT_DIR = DATA_PATH / '06-feature-importance'       # output of step 06
LOCAL_INPUT = Path('/content/input')                # local copy of the input files

Mounted at /content/drive


In [3]:
# Copy the inputs from Drive to the Colab disk: reading a local file is much faster.
import shutil

LOCAL_INPUT.mkdir(exist_ok=True)
for name in ['config.json', 'dataset.parquet', 'edges.parquet']:
    source, target = INPUT_DIR / name, LOCAL_INPUT / name
    assert source.exists(), f'{source} is missing: run step 02 first'
    if not target.exists() or target.stat().st_size != source.stat().st_size:
        shutil.copy(source, target)
    print(f'{name}: {target.stat().st_size / 1e6:,.1f} MB')

config.json: 0.1 MB
dataset.parquet: 76.1 MB
edges.parquet: 2.2 MB


In [4]:
NOTEBOOK = 'gpu'
MODELS = ['xgb', 'mlp', 'sage', 'gatv2', 'cheb', 'gin']   # the models explained by this notebook
ONLY_PAIRS = None              # None = every pair; or a list such as ['transparent__sage'] to run only some pairs
SETTINGS = {
    'sample': {'seed': 0, 'licit': 1000, 'core_illicit': 60, 'core_licit': 20},   # the explain sample (same in both notebooks)
    'shuffles': 5,             # permutations per column (seeds 0-4)
    'ig_steps': 50,            # points of the path of Integrated Gradients
    'ig_gnn_seeds': 'auto',    # seeds of the GNNs explained with Integrated Gradients: 'auto' = [0, 1, 2] if the budget allows, else [0]
    'ig_budget_hours': 2.5,
    'matrix_seed': 0,          # the seed whose full matrix of attributions is saved
    'rf_licit': 1000, 'rf_shap_seeds': [0, 1, 2],   # not used here (CPU notebook)
    'xgb_device': 'cuda',
}
GNNEXPLAINER = {'model_seed': 0, 'explainer_seed': 0, 'epochs': 200, 'lr': 0.01, 'extra_fp': 20, 'fp_seed': 0,
                'stability_nodes': 20}
FORCE_RECOMPUTE = False        # True = delete the results of this notebook's models and compute them again

## 1. Shared code

Data, seeds and metrics (1/4), classical models (2/4) and neural models (3/4): identical to the tuning and final-run notebooks. Then the importance code (identical in the two computing notebooks of this step) and the GNNExplainer code (this notebook only).

In [5]:
# ==== SHARED CODE 1/4 · data, seeds, metrics · SHARED_CODE_VERSION 1 ====
# Identical copies in: 03-tuning-cpu, 03-tuning-gpu, and later 04-final-runs-cpu, 04-final-runs-gpu.
# Change every copy together and raise SHARED_CODE_VERSION.
import json
import os
import random
import sys
import time
from datetime import datetime, timezone
from types import SimpleNamespace

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')   # deterministic cuBLAS: must be set before PyTorch uses the GPU
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

SHARED_CODE_VERSION = 1
CONFIG_VERSION = 1                 # the version of config.json this code was written for
DETERMINISTIC_ALGORITHMS = True    # ask PyTorch for deterministic GPU kernels (only a warning where none exists)


def set_seed(seed):
    """Fix every random generator: Python, NumPy and, when it is loaded, PyTorch (CPU, every GPU, cuDNN)."""
    random.seed(seed)
    np.random.seed(seed)
    if 'torch' in sys.modules:
        import torch
        torch.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(DETERMINISTIC_ALGORITHMS, warn_only=True)


def atomic(path, write):
    """Write through a temporary file, then rename it: a run that stops while writing
    never leaves a half-written file on Drive."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    write(tmp)
    os.replace(tmp, path)


def to_builtin(value):
    """Make NumPy numbers JSON-serializable."""
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f'not JSON-serializable: {type(value)}')


def write_csv(df, path):
    atomic(path, lambda tmp: df.to_csv(tmp, index=False, float_format='%.10g'))


def write_json(obj, path):
    text = json.dumps(obj, indent=1, default=to_builtin, ensure_ascii=False)
    atomic(path, lambda tmp: tmp.write_text(text, encoding='utf-8'))


def fmt_time(seconds):
    """Format a duration as '12.3s', '4m 05s' or '1h 07m'."""
    if seconds < 60:
        return f'{seconds:.1f}s'
    minutes, seconds = divmod(int(seconds), 60)
    if minutes < 60:
        return f'{minutes}m {seconds:02d}s'
    hours, minutes = divmod(minutes, 60)
    return f'{hours}h {minutes:02d}m'


def load_inputs(folder):
    """config.json, dataset.parquet and edges.parquet of step 02, with the arrays every later step needs."""
    config = json.loads((folder / 'config.json').read_text(encoding='utf-8'))
    assert config['version'] == CONFIG_VERSION, f"config.json has version {config['version']}, this code expects {CONFIG_VERSION}"
    dataset = pd.read_parquet(folder / 'dataset.parquet')
    edges = pd.read_parquet(folder / 'edges.parquet')
    assert len(dataset) == config['files']['n_rows'] and len(edges) == config['files']['n_edges']
    y = dataset['y'].to_numpy(dtype=np.float64, na_value=np.nan)
    return SimpleNamespace(
        config=config, dataset=dataset,
        split=dataset['split'].to_numpy(), in_eval=dataset['in_eval'].to_numpy(dtype=bool), illicit=(y == 1),
        src=edges['src_row'].to_numpy(np.int64), dst=edges['dst_row'].to_numpy(np.int64))


def scale(values, spec):
    """One column of the profile `scaled` (config['transforms']): z = (t - mean) / std, with
    t = sign(x) log(1 + |x|) for the two log transformations and t = x otherwise (binary: mean 0, std 1)."""
    x = np.asarray(values, dtype=np.float64)
    if spec['transform'] in ('log1p+z', 'signed_log1p+z'):
        x = np.sign(x) * np.log1p(np.abs(x))
    return (x - spec['mean']) / spec['std']


def feature_matrix(data, menu, profile):
    """The input matrix of a menu for every node (float32): raw values (profile `raw`) or transformed
    (profile `scaled`). The menu `none` gives the constant input x = 1."""
    columns = data.config['menus'][menu]['columns']
    if not columns:
        return np.ones((len(data.dataset), 1), dtype=np.float32)
    X = data.dataset[columns].to_numpy(np.float64)
    if profile == 'scaled':
        specs = data.config['transforms']['profiles']['scaled']['columns']
        for k, col in enumerate(columns):
            X[:, k] = scale(X[:, k], specs[col])
    return X.astype(np.float32)


def eval_rows(data, split):
    """Row numbers of the evaluation rows of a split (labeled and with the Elliptic++ data)."""
    return np.flatnonzero((data.split == split) & data.in_eval)


def suggest_params(trial, space):
    """Ask an Optuna trial for one value of every hyper-parameter of a search space of config.json."""
    params = {}
    for name, spec in space.items():
        if spec['type'] == 'categorical':
            params[name] = trial.suggest_categorical(name, spec['choices'])
        elif spec['type'] == 'float':
            params[name] = trial.suggest_float(name, spec['low'], spec['high'], log=spec['log'])
        elif spec['type'] == 'int':
            params[name] = trial.suggest_int(name, spec['low'], spec['high'], log=spec['log'])
        else:
            raise ValueError(f'unknown type {spec["type"]} for {name}')
    return params

In [6]:
# ==== SHARED CODE 2/4 · classical models · SHARED_CODE_VERSION 1 ====
# Identical copies in: 03-tuning-cpu, 03-tuning-gpu, and later 04-final-runs-cpu, 04-final-runs-gpu.
# Change every copy together and raise SHARED_CODE_VERSION.
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# XGBoost on the GPU with NumPy inputs prints this at every prediction; it is harmless.
warnings.filterwarnings('ignore', message='.*Falling back to prediction using DMatrix.*')


def fit_classical(model_name, params, fixed, X_train, y_train, X_val, y_val, seed, final=False):
    """Train Logistic Regression, Random Forest or XGBoost on the train rows.
    XGBoost stops adding trees when the validation PR-AUC stops improving (early stopping).
    `final=True` (step 04) also computes the out-of-bag score of the Random Forest."""
    fixed = dict(fixed)
    info = {}
    start = time.perf_counter()
    if model_name == 'logreg':
        model = LogisticRegression(**fixed, **params, random_state=seed)
        model.fit(X_train, y_train)
    elif model_name == 'rf':
        oob = bool(fixed.pop('oob_score_in_final_runs', False)) and final
        model = RandomForestClassifier(**fixed, **params, oob_score=oob, random_state=seed)
        model.fit(X_train, y_train)
        if oob:
            info['oob_average_precision'] = float(average_precision_score(
                y_train, np.nan_to_num(model.oob_decision_function_[:, 1])))
    elif model_name == 'xgb':
        model = XGBClassifier(**fixed, **params, random_state=seed)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        info['best_iteration'] = int(model.best_iteration)
    else:
        raise ValueError(model_name)
    info['seconds'] = time.perf_counter() - start
    scores = model.predict_proba(X_val)[:, 1]
    return SimpleNamespace(model=model, val_scores=scores, val_ap=float(average_precision_score(y_val, scores)), info=info)

In [7]:
# ==== SHARED CODE 3/4 · neural models · SHARED_CODE_VERSION 1 ====
# Identical copies in: 03-tuning-gpu, and later 04-final-runs-gpu and 05-plots (to recompute embeddings).
# Change every copy together and raise SHARED_CODE_VERSION.
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import ChebConv, GATv2Conv, GINConv, JumpingKnowledge, SAGEConv
from torch_geometric.utils import to_undirected

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def period_data(data, X, split, kind):
    """Tensors of one split on the GPU.
    kind 'graph' (GNNs): every node of the time steps of the split (unknown included), the edges among them made
    undirected; loss and metrics only on the evaluation rows (`mask`).
    kind 'rows' (MLP): only the evaluation rows of the split, no edges."""
    if kind == 'rows':
        rows = eval_rows(data, split)
        y = data.illicit[rows]
        return SimpleNamespace(
            x=torch.tensor(X[rows], device=DEVICE), edge_index=None,
            y=torch.tensor(y, dtype=torch.float32, device=DEVICE),
            mask=torch.ones(len(rows), dtype=torch.bool, device=DEVICE), y_eval=y.astype(int), rows=rows)
    nodes = np.flatnonzero(data.split == split)
    local = np.full(len(data.split), -1)
    local[nodes] = np.arange(len(nodes))
    keep = local[data.src] >= 0
    assert (local[data.dst[keep]] >= 0).all(), 'an edge leaves the split'   # no edge links two time steps
    edge_index = torch.tensor(np.vstack([local[data.src[keep]], local[data.dst[keep]]]), dtype=torch.long)
    mask = data.in_eval[nodes]
    y = data.illicit[nodes]
    return SimpleNamespace(
        x=torch.tensor(X[nodes], device=DEVICE),
        edge_index=to_undirected(edge_index, num_nodes=len(nodes)).to(DEVICE),
        y=torch.tensor(y, dtype=torch.float32, device=DEVICE),
        mask=torch.tensor(mask, device=DEVICE), y_eval=y[mask].astype(int), rows=nodes[mask])


class MLP(nn.Module):
    """`layers` hidden layers of width `hidden` (ReLU, dropout), then one output."""

    def __init__(self, d_in, hidden, layers, dropout):
        super().__init__()
        dims = [d_in] + [hidden] * layers
        self.hidden = nn.ModuleList(nn.Linear(a, b) for a, b in zip(dims[:-1], dims[1:]))
        self.out = nn.Linear(hidden, 1)
        self.dropout = dropout

    def forward(self, x, edge_index=None, return_hidden=False):
        for layer in self.hidden:
            x = F.dropout(F.relu(layer(x)), self.dropout, self.training)
        return x if return_hidden else self.out(x).squeeze(-1)


def make_conv(kind, d_in, d_out, args):
    if kind == 'sage':
        return SAGEConv(d_in, d_out, aggr=args['aggr'])
    if kind == 'gatv2':
        heads = args['heads'] if d_out > 1 else 1
        return GATv2Conv(d_in, d_out // heads, heads=heads, dropout=args['att_dropout'])
    if kind == 'cheb':
        return ChebConv(d_in, d_out, K=args['K'], normalization=args.get('normalization', 'sym'))
    if kind == 'gin':
        return GINConv(nn.Sequential(nn.Linear(d_in, d_out), nn.ReLU(), nn.Linear(d_out, d_out)),
                       train_eps=args['train_eps'])
    raise ValueError(kind)


class GNN(nn.Module):
    """`layers` graph convolutions in one of three variants (Ferretti, D'Angelo and Ghini, 2025):
    'base' (the last convolution gives the output), 'lin' (a linear layer after the convolutions),
    'skip' (like 'lin', and the output of the first convolution is added to the output of the last one).
    `jumping_knowledge` ('max' or 'cat', GIN only) combines the outputs of every convolution before the linear layer."""

    def __init__(self, kind, d_in, hidden, layers, variant, dropout, jumping_knowledge=None, **conv_args):
        super().__init__()
        dims = [d_in] + [hidden] * layers
        if variant == 'base':
            dims[-1] = 1
        self.convs = nn.ModuleList(make_conv(kind, a, b, conv_args) for a, b in zip(dims[:-1], dims[1:]))
        self.jk = None if jumping_knowledge is None else JumpingKnowledge(jumping_knowledge)
        width = hidden * layers if jumping_knowledge == 'cat' else hidden
        self.lin = None if variant == 'base' else nn.Linear(width, 1)
        self.variant, self.dropout = variant, dropout

    def forward(self, x, edge_index, return_hidden=False):
        outputs = []
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if self.lin is not None or i < len(self.convs) - 1:
                x = F.dropout(F.relu(x), self.dropout, self.training)
            outputs.append(x)
        if self.variant == 'skip':
            x = x + outputs[0]
        if self.jk is not None:
            x = self.jk(outputs)
        if return_hidden:
            return x
        if self.lin is not None:
            x = self.lin(x)
        return x.squeeze(-1)


def build_neural(model_name, params, d_in, fixed=None):
    """The network of a model from its hyper-parameters (lr and weight_decay belong to the optimizer)."""
    p = {k: v for k, v in params.items() if k not in ('lr', 'weight_decay')}
    fixed = dict(fixed or {})
    if model_name == 'mlp':
        net = MLP(d_in, **p)
    elif model_name == 'gatv2':
        heads, per_head = p.pop('heads'), p.pop('hidden_per_head')
        net = GNN('gatv2', d_in, hidden=heads * per_head, heads=heads, **p)
    elif model_name == 'gin':
        net = GNN('gin', d_in, variant='lin', **p)
    else:
        net = GNN(model_name, d_in, **p, **fixed)
    return net.to(DEVICE)


def predict(net, part):
    """Scores (probability of illicit) of the evaluation rows of one split."""
    net.eval()
    with torch.no_grad():
        return torch.sigmoid(net(part.x, part.edge_index)[part.mask]).float().cpu().numpy()


def train_neural(net, train, val, lr, weight_decay, pos_weight, max_epochs, patience,
                 report=None, curves=False, train_ap_every=None, keep_best_state=False):
    """Full-batch training with Adam and binary cross-entropy weighted by `pos_weight`.
    After every epoch: PR-AUC on the validation rows; training stops after `patience` epochs without improvement,
    and the result is the best epoch. `report(epoch, best_ap)` is called after every epoch (pruning of the tuning).
    `curves` keeps one row per epoch (train and validation loss, validation PR-AUC, train PR-AUC every
    `train_ap_every` epochs); `keep_best_state` keeps a copy of the weights of the best epoch."""
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE))
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    best_ap, best_epoch, best_state, history, diverged = 0.0, 0, None, [], False
    start = time.perf_counter()
    for epoch in range(1, max_epochs + 1):
        net.train()
        optimizer.zero_grad()
        out = net(train.x, train.edge_index)
        loss = loss_fn(out[train.mask], train.y[train.mask])
        loss.backward()
        optimizer.step()
        net.eval()
        with torch.no_grad():
            val_out = net(val.x, val.edge_index)[val.mask]
        if not torch.isfinite(val_out).all():   # the training diverged (for example, a learning rate too high)
            diverged = True
            break
        val_ap = float(average_precision_score(val.y_eval, val_out.float().cpu().numpy()))
        if curves:
            row = {'epoch': epoch, 'train_loss': float(loss), 'val_loss': float(loss_fn(val_out, val.y[val.mask])),
                   'val_average_precision': val_ap}
            if train_ap_every and epoch % train_ap_every == 0:
                row['train_average_precision'] = float(average_precision_score(train.y_eval, predict(net, train)))
            history.append(row)
        if val_ap > best_ap:
            best_ap, best_epoch = val_ap, epoch
            if keep_best_state:
                best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        if report is not None:
            report(epoch, best_ap)
        if epoch - best_epoch >= patience:
            break
    return {'best_ap': best_ap, 'best_epoch': best_epoch, 'epochs': epoch, 'diverged': diverged,
            'hit_max_epochs': (not diverged) and epoch == max_epochs and epoch - best_epoch < patience,
            'seconds': time.perf_counter() - start, 'n_params': sum(p.numel() for p in net.parameters()),
            'peak_gpu_mb': torch.cuda.max_memory_allocated() / 1e6 if DEVICE.type == 'cuda' else None,
            'curves': history, 'best_state': best_state}

In [8]:
# ==== IMPORTANCE CODE · explain sample, shuffles, permutation, attributions, files · identical in 06-importance-cpu and 06-importance-gpu ====
# Change both copies together. It uses the shared code above (blocks 1 and 2, and block 3 for the networks).
# PyTorch and SHAP are imported only inside the functions that need them.
import gc
import math

import joblib
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

SPLITS = ('train', 'val', 'test')
FAMILIES = ['LF', 'AF', 'A', 'B', 'C', 'D']
GNN_MODELS = ('sage', 'gatv2', 'cheb', 'gin')


def triple_id(menu, model, seed):
    return f'{menu}__{model}__seed{seed}'


def free_memory():
    gc.collect()
    if 'torch' in sys.modules:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def sigmoid64(z):
    """The score in float64: the float32 sigmoid would round every logit above about 17 to exactly 1 (ties)."""
    with np.errstate(over='ignore'):
        return 1 / (1 + np.exp(-np.asarray(z, dtype=np.float64)))


# ---------- families ----------
def column_family(config):
    """Every kept column -> its family (LF, AF, A, B, C, D), as decided in step 02."""
    return {col: fam for fam in FAMILIES for col in config['columns']['families'][fam]['kept']}


def menu_units(config, menu, families_only=False):
    """What is shuffled together: every column alone and, if the menu has two families or more, every family
    (a menu with one family gets no family value: shuffling the family would shuffle the whole menu).
    Returns {name: (level, family, positions of the columns)} and {family: positions}."""
    columns = config['menus'][menu]['columns']
    fam = column_family(config)
    groups = {}
    for k, col in enumerate(columns):
        groups.setdefault(fam[col], []).append(k)
    units = {} if families_only else {col: ('column', fam[col], [k]) for k, col in enumerate(columns)}
    if len(groups) > 1:
        units.update({f: ('family', f, ks) for f, ks in groups.items()})
    return units, groups


# ---------- the explain sample ----------
def largest_remainder(total, counts):
    """Split `total` into integers proportional to `counts` (largest remainders; ties go to the first)."""
    counts = np.asarray(counts, dtype=np.float64)
    quota = total * counts / counts.sum()
    base = np.floor(quota).astype(int)
    order = np.argsort(-(quota - base), kind='stable')
    base[order[:total - base.sum()]] += 1
    return base


def draw_by_time_step(rng, rows, steps_of_rows, total):
    """`total` of `rows` drawn without replacement, in proportion to the time steps."""
    steps = np.unique(steps_of_rows)
    counts = largest_remainder(total, [(steps_of_rows == t).sum() for t in steps])
    picked = [rng.choice(rows[steps_of_rows == t], size=k, replace=False) for t, k in zip(steps, counts) if k]
    return np.sort(np.concatenate(picked)) if picked else np.array([], dtype=np.int64)


def build_explain_sample(data, settings):
    """One explain sample for every model: every illicit transaction of the test plus `licit` licit ones drawn in
    proportion to the time steps. weight = how many test transactions a row stands for (illicit 1, licit
    9,342 / 1,000), so weighted means over the sample estimate means over the test. Inside it, the core of
    GNNExplainer: `core_illicit` illicit and `core_licit` licit rows, also in proportion to the time steps.
    `licit_order` is a fixed random order of the licit rows: a smaller subset takes the first ones."""
    rng = np.random.default_rng(settings['seed'])
    rows = eval_rows(data, 'test')
    step = data.dataset['time_step'].to_numpy()
    illicit = data.illicit[rows]
    licit_rows = draw_by_time_step(rng, rows[~illicit], step[rows[~illicit]], settings['licit'])
    illicit_rows = rows[illicit]
    core = np.concatenate([draw_by_time_step(rng, illicit_rows, step[illicit_rows], settings['core_illicit']),
                           draw_by_time_step(rng, licit_rows, step[licit_rows], settings['core_licit'])])
    sample = np.concatenate([illicit_rows, licit_rows])
    order = np.full(len(sample), -1)
    order[len(illicit_rows):] = rng.permutation(len(licit_rows))
    n_licit_test = int((~illicit).sum())
    table = pd.DataFrame({'row': sample, 'txId': data.dataset['txId'].to_numpy()[sample], 'time_step': step[sample],
                          'y': data.illicit[sample].astype(int),
                          'weight': np.where(data.illicit[sample], 1.0, n_licit_test / len(licit_rows)),
                          'licit_order': order, 'gnnexplainer_core': np.isin(sample, core)})
    return table.sort_values('row', kind='stable').reset_index(drop=True)


def load_or_build_sample(data, settings, path):
    """The first notebook that runs writes the sample; the other one rebuilds it and checks that it is identical."""
    table = build_explain_sample(data, settings)
    assert abs(table['weight'].sum() - len(eval_rows(data, 'test'))) < 1e-6, 'the weights must add up to the test size'
    if path.exists():
        saved = pd.read_csv(path)
        same = (np.array_equal(saved['row'], table['row']) and np.allclose(saved['weight'], table['weight'])
                and np.array_equal(saved['licit_order'], table['licit_order'])
                and np.array_equal(saved['gnnexplainer_core'].astype(bool), table['gnnexplainer_core']))
        assert same, f'{path.name} differs from the sample rebuilt here: different settings or data'
    else:
        write_csv(table, path)
    return table


def sample_subset(sample, n_licit):
    """All illicit rows and the first `n_licit` licit rows of the fixed random order, re-weighted to stand for the test."""
    keep = (sample['y'] == 1) | ((sample['licit_order'] >= 0) & (sample['licit_order'] < n_licit))
    sub = sample[keep].copy()
    sub.loc[sub['y'] == 0, 'weight'] = sample.loc[sample['y'] == 0, 'weight'].sum() / n_licit
    return sub.reset_index(drop=True)


# ---------- permutation importance ----------
def within_step_permutations(steps_of_positions, n_shuffles, first_seed=0):
    """One permutation of the positions per shuffle r (seed first_seed + r): values move only between positions of the
    same time step. The same seeds and the same positions give the same permutations to every model."""
    perms = []
    for r in range(n_shuffles):
        rng = np.random.default_rng(first_seed + r)
        perm = np.arange(len(steps_of_positions))
        for t in np.unique(steps_of_positions):
            idx = np.flatnonzero(steps_of_positions == t)
            perm[idx] = idx[rng.permutation(len(idx))]
        perms.append(perm)
    return perms


def permutation_importance(score, X, y, units, perms):
    """Drop of PR-AUC and ROC-AUC when the columns of a unit are shuffled and the model is NOT retrained.
    The columns of a unit (one column, or a whole family) move with the same permutation, so a row keeps its own
    combination of their values. `score(X)` gives the scores of the evaluation rows; X is a NumPy array or a PyTorch
    tensor, changed in place and restored after every unit."""
    base = score(X)
    base_ap, base_roc = average_precision_score(y, base), roc_auc_score(y, base)
    is_tensor = not isinstance(X, np.ndarray)
    if is_tensor:
        import torch
        perms = [torch.as_tensor(np.asarray(p, dtype=np.int64), device=X.device) for p in perms]
    rows = []
    for name, (level, family, positions) in units.items():
        cols = torch.as_tensor(positions, dtype=torch.long, device=X.device) if is_tensor else np.asarray(positions)
        original = X[:, cols].clone() if is_tensor else X[:, cols].copy()
        ap, roc = [], []
        for perm in perms:
            X[:, cols] = original[perm]
            s = score(X)
            ap.append(base_ap - average_precision_score(y, s))
            roc.append(base_roc - roc_auc_score(y, s))
        X[:, cols] = original
        spread = (lambda v: float(np.std(v, ddof=1))) if len(perms) > 1 else (lambda v: 0.0)
        rows.append({'level': level, 'name': name, 'family': family, 'n_columns': len(positions),
                     'ap_drop': float(np.mean(ap)), 'ap_drop_std': spread(ap),
                     'roc_drop': float(np.mean(roc)), 'roc_drop_std': spread(roc),
                     'base_ap': base_ap, 'base_roc': base_roc, 'shuffles': len(perms)})
    restored = score(X)
    return pd.DataFrame(rows), float(np.max(np.abs(restored - base)))


# ---------- attributions ----------
def attribution_summary(phi, values, y, weights, columns, families, groups, neighbours=None):
    """Summary of a matrix of attributions phi (rows x columns) over rows that stand for the test with `weights`.
    For every column, every family (menus with two families or more) and the whole menu: weighted mean of |phi|
    (the test), mean |phi| over the illicit rows, weighted mean of phi, share of the total, density
    (share / share of the columns), and for the columns the direction: Spearman correlation between the value of
    the column and its phi (> 0: high values push towards "illicit"). `neighbours` (GNNs): the part of phi that comes
    from the features of the neighbours; neighbour_share = mean |neighbour part| / (mean |own part| + mean |neighbour part|)."""
    w = np.asarray(weights, dtype=np.float64)
    w = w / w.sum()
    illicit = np.asarray(y).astype(bool)
    n = len(columns)

    def one(p, nb=None):
        out = {'mean_abs': float(w @ np.abs(p)),
               'mean_abs_illicit': float(np.abs(p[illicit]).mean()) if illicit.any() else np.nan,
               'mean': float(w @ p)}
        if nb is not None:
            own_abs, nb_abs = float(w @ np.abs(p - nb)), float(w @ np.abs(nb))
            out['neighbour_share'] = nb_abs / (own_abs + nb_abs) if own_abs + nb_abs > 0 else np.nan
        return out

    rows = []
    for j, col in enumerate(columns):
        row = {'level': 'column', 'name': col, 'family': families[j], 'n_columns': 1,
               **one(phi[:, j], None if neighbours is None else neighbours[:, j])}
        x, p = values[:, j], phi[:, j]
        ok = np.isfinite(x)
        row['direction'] = (float(spearmanr(x[ok], p[ok]).correlation)
                            if ok.sum() > 2 and np.ptp(x[ok]) > 0 and np.ptp(p[ok]) > 0 else np.nan)
        rows.append(row)
    total = sum(r['mean_abs'] for r in rows)
    for r in rows:
        r['share'] = r['mean_abs'] / total if total > 0 else np.nan
        r['density'] = r['share'] * n
    if len(groups) > 1:
        family_rows = []
        for fam, positions in groups.items():
            family_rows.append({'level': 'family', 'name': fam, 'family': fam, 'n_columns': len(positions),
                                **one(phi[:, positions].sum(axis=1),
                                      None if neighbours is None else neighbours[:, positions].sum(axis=1)),
                                'direction': np.nan})
        family_total = sum(r['mean_abs'] for r in family_rows)
        for r in family_rows:
            r['share'] = r['mean_abs'] / family_total if family_total > 0 else np.nan
            r['density'] = r['share'] / (r['n_columns'] / n)
        rows += family_rows
    rows.append({'level': 'menu', 'name': 'all', 'family': 'all', 'n_columns': n,
                 **one(phi.sum(axis=1), None if neighbours is None else neighbours.sum(axis=1)),
                 'direction': np.nan, 'share': 1.0, 'density': 1.0})
    return pd.DataFrame(rows)


def linear_shap(model, X, mean):
    """SHAP of a logistic regression in log-odds (exact for a linear model, features taken as independent):
    phi_j = w_j (x_j - mean_j); base = b + w . mean; base + sum_j phi_j = the logit."""
    w = model.coef_[0].astype(np.float64)
    phi = (X.astype(np.float64) - mean) * w
    return phi, float(model.intercept_[0] + w @ mean)


def tree_shap_forest(model, X):
    """TreeSHAP (exact, path-dependent) of a scikit-learn Random Forest, in probability of "illicit"."""
    import shap
    explainer = shap.TreeExplainer(model)
    values = explainer.shap_values(X, check_additivity=False)
    if isinstance(values, list):
        values = values[1]
    elif np.ndim(values) == 3:
        values = values[:, :, 1]
    base = np.ravel(explainer.expected_value)
    return np.asarray(values, dtype=np.float64), float(base[1] if len(base) > 1 else base[0])


def integrated_gradients_rows(net, x, baseline, steps, chunk=8192):
    """Integrated Gradients of the logit of every row of an MLP (Sundararajan et al., 2017), with the midpoint rule:
    phi_j = (x_j - x'_j) * (1/m) sum_k d f(x' + a_k (x - x')) / d x_j, with a_k = (k + 1/2) / m.
    Completeness: sum_j phi_j = f(x) - f(x'), up to the error of the rule (measured and returned)."""
    import torch
    net.eval()
    out, errors, logits = [], [], []
    for start in range(0, x.shape[0], chunk):
        xc = x[start:start + chunk]
        total = torch.zeros_like(xc)
        for k in range(steps):
            point = (baseline + (k + 0.5) / steps * (xc - baseline)).detach().requires_grad_(True)
            grad, = torch.autograd.grad(net(point).sum(), point)
            total += grad
        attr = ((xc - baseline) * total / steps).double()
        with torch.no_grad():
            f_x = net(xc).double()
            f_base = net(baseline.expand_as(xc)).double()
        gap = (f_x - f_base)
        errors.append(((attr.sum(dim=1) - gap).abs() / gap.abs().clamp(min=1e-6)).cpu().numpy())
        out.append(attr.cpu().numpy())
        logits.append(f_x.cpu().numpy())
    return np.vstack(out), float(f_base[0]), np.concatenate(logits), np.concatenate(errors)


def receptive_hops(model, params):
    """How far a GNN looks: one hop per layer, K - 1 hops per layer for ChebNet (polynomials up to order K - 1)."""
    return params['layers'] * (params['K'] - 1) if model == 'cheb' else params['layers']


def gnn_integrated_gradients(net, part, targets, baseline, hops, steps, max_nodes=400_000):
    """Integrated Gradients of the logit of every target node with respect to the features of EVERY node of its
    neighbourhood. The neighbourhood is the subgraph within hops + 1 hops (the extra hop keeps the degree of the outer
    nodes, which ChebNet's normalisation uses); the logit on it must equal the logit on the whole graph. The m points of
    the path are m copies of the subgraph in one disjoint batch. For each target: the attribution of its own features
    (own) and the sum over the other nodes (neighbours), both of length d; total = own + neighbours."""
    import torch
    from torch_geometric.utils import k_hop_subgraph
    net.eval()
    with torch.no_grad():
        full = net(part.x, part.edge_index).double().cpu().numpy()
    n_all, d = part.x.shape
    own, nbr, err, sub_gap, sizes = [], [], [], [], []
    for t in targets:
        subset, ei, mapping, _ = k_hop_subgraph(int(t), hops + 1, part.edge_index, relabel_nodes=True, num_nodes=n_all)
        x_sub = part.x[subset]
        n, centre = len(subset), int(mapping[0])
        grads = torch.zeros_like(x_sub)
        per_batch = max(1, min(steps, max_nodes // max(n, 1)))
        for start in range(0, steps, per_batch):
            m = min(per_batch, steps - start)
            alphas = (torch.arange(start, start + m, device=x_sub.device, dtype=x_sub.dtype) + 0.5) / steps
            points = (baseline + alphas[:, None, None] * (x_sub - baseline)[None]).reshape(m * n, d)
            points = points.detach().requires_grad_(True)
            batch_edges = torch.cat([ei + j * n for j in range(m)], dim=1)
            out = net(points, batch_edges).view(m, n)[:, centre].sum()
            grad, = torch.autograd.grad(out, points)
            grads += grad.view(m, n, d).sum(dim=0)
        attr = ((x_sub - baseline) * grads / steps).double()
        with torch.no_grad():
            logit = float(net(x_sub, ei)[centre])
            logit_base = float(net(baseline.expand_as(x_sub).contiguous(), ei)[centre])
        a_own = attr[centre].cpu().numpy()
        a_all = attr.sum(dim=0).cpu().numpy()
        own.append(a_own)
        nbr.append(a_all - a_own)
        gap = logit - logit_base
        err.append(abs(a_all.sum() - gap) / max(abs(gap), 1e-6))
        sub_gap.append(abs(logit - full[t]))
        sizes.append(n)
    own, nbr = np.vstack(own), np.vstack(nbr)
    return SimpleNamespace(total=own + nbr, neighbours=nbr, completeness=np.array(err),
                           subgraph_difference=np.array(sub_gap), sizes=np.array(sizes), logits=full[np.asarray(targets)])


# ---------- models of step 04 ----------
def load_classical(model, path, xgb_device='cuda'):
    if model in ('logreg', 'rf'):
        return joblib.load(path)
    m = XGBClassifier()
    m.load_model(str(path))
    m.get_booster().set_param({'device': xgb_device})
    return m


def classical_score(model, m, record):
    """The score function of a classical model, in float64. XGBoost uses the trees up to its best iteration."""
    if model == 'xgb':
        booster, best = m.get_booster(), (0, int(record['training']['best_iteration']) + 1)
        return lambda X: sigmoid64(booster.inplace_predict(X, iteration_range=best, predict_type='margin'))
    return lambda X: m.predict_proba(X)[:, 1].astype(np.float64)


def load_network(model, record, path):
    import torch
    net = build_neural(model, record['params'], record['d_in'], record['fixed'])
    net.load_state_dict(torch.load(path, map_location=DEVICE))
    net.eval()
    return net


def network_score(net, edge_index, mask):
    import torch

    def score(x):
        with torch.no_grad():
            return sigmoid64(net(x, edge_index)[mask].double().cpu().numpy())
    return score


# ---------- files, checks, progress ----------
class Progress:
    """Log lines with triples done and left and an estimate of the time left (expected work of each triple,
    corrected by the speed measured in this session). Lines are also appended to a log file."""

    def __init__(self, expected, done, log_path):
        self.expected, self.done, self.log_path = expected, set(done), log_path
        self.work, self.start, self.lines = 0.0, time.time(), []

    def finish(self, key):
        self.done.add(key)
        self.work += self.expected[key]

    def status(self):
        elapsed = time.time() - self.start
        speed = elapsed / self.work if self.work > 0 else None
        left = sum(s for t, s in self.expected.items() if t not in self.done)
        eta = fmt_time(left * speed) if speed else 'measuring'
        return f'done {len(self.done)}, left {len(self.expected) - len(self.done)} · elapsed {fmt_time(elapsed)}, left ~{eta}'

    def print(self, line):
        print(line, flush=True)
        self.lines.append(f'{datetime.now():%Y-%m-%d %H:%M:%S} {line}')
        self.flush()

    def flush(self):
        if self.lines and self.log_path is not None:
            self.log_path.parent.mkdir(parents=True, exist_ok=True)
            with open(self.log_path, 'a', encoding='utf-8') as f:
                f.write('\n'.join(self.lines) + '\n')
        self.lines = []


def add_checks(path, rows):
    """Append rows (triple, check, value, limit, ok) to checks.csv, replacing older rows of the same check."""
    old = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=['triple', 'check', 'value', 'limit', 'ok'])
    new = pd.DataFrame(rows)
    keys = set(zip(new['triple'], new['check']))
    old = old[[(t, c) not in keys for t, c in zip(old['triple'], old['check'])]]
    write_csv(pd.concat([old, new], ignore_index=True), path)


def expected_work(config, menu, model, settings):
    """Relative work of one triple, for the estimate of the time left (the speed is measured while running)."""
    n = len(config['menus'][menu]['columns'])
    units, groups = menu_units(config, menu)
    n_fam = len(groups) if len(groups) > 1 else 0
    predicts = (len(units) + n_fam) * settings['shuffles']
    per_predict = {'logreg': 0.01, 'rf': 1.0, 'xgb': 0.03, 'mlp': 0.01, 'sage': 0.05, 'gatv2': 0.1, 'cheb': 0.08,
                   'gin': 0.05}[model]
    licit = settings['rf_licit'] if isinstance(settings['rf_licit'], int) else 1000
    attribution = {'logreg': 0.1, 'rf': 0.004 * n * (524 + licit), 'xgb': 1, 'mlp': 1}.get(
        model, 1524 * 0.02 * max(n, 1) / 50)
    return 1 + predicts * per_predict + attribution


def explain_triple(ctx, menu, model, seed):
    """Every measure of one triple, then its files. done/<triple>.json is written last: "triple finished"."""
    data, config, dirs, settings = ctx.data, ctx.data.config, ctx.dirs, ctx.settings
    tid = triple_id(menu, model, seed)
    record = ctx.records[(menu, model, seed)]
    start = time.perf_counter()
    columns = config['menus'][menu]['columns']
    fam = column_family(config)
    families = [fam[c] for c in columns]
    units, groups = menu_units(config, menu)
    family_units, _ = menu_units(config, menu, families_only=True)
    step = data.dataset['time_step'].to_numpy()
    rows = {split: eval_rows(data, split) for split in SPLITS}
    y = {split: data.illicit[rows[split]].astype(int) for split in SPLITS}
    saved = pd.read_parquet(dirs.runs / 'predictions' / f'{tid}.parquet')
    saved = saved[saved['split'] == 'test']
    assert np.array_equal(saved['row'].to_numpy(), rows['test']), 'the saved predictions are in another order'
    X_all = feature_matrix(data, menu, config['models'][model]['profile'])   # the menu none gives x = 1
    values_of = lambda used: data.dataset.iloc[used][columns].to_numpy(np.float64)   # raw values, for the direction
    model_path = dirs.runs / record['files']['model']
    checks, timings, tables = [], {}, []
    method, matrix = None, None

    def check(name, value, limit):
        checks.append({'triple': tid, 'check': name, 'value': float(value), 'limit': limit, 'ok': bool(value <= limit)})
        assert value <= limit, f'{tid}: {name} = {value:.3g} > {limit:g}'

    if model in ('logreg', 'rf', 'xgb'):
        m = load_classical(model, model_path, settings.get('xgb_device', 'cuda'))
        score = classical_score(model, m, record)
        X = {split: X_all[rows[split]].copy() for split in ('test', 'val')}
        test_scores = score(X['test'])
        tolerance = 1e-6 if model == 'xgb' else 1e-12
        check('reload: largest score difference', np.max(np.abs(test_scores - saved['score'].to_numpy())), tolerance)
        t0 = time.perf_counter()
        perm_rows = {split: within_step_permutations(step[rows[split]], settings['shuffles']) for split in ('test', 'val')}
        table, restored = permutation_importance(score, X['test'], y['test'], units, perm_rows['test'])
        tables.append(table.assign(data='test'))
        if family_units:
            table, _ = permutation_importance(score, X['val'], y['val'], family_units, perm_rows['val'])
            tables.append(table.assign(data='val'))
        check('permutation: scores restored', restored, 1e-9)
        timings['permutation'] = time.perf_counter() - t0
        t0 = time.perf_counter()
        if model == 'logreg':
            mean = X_all[rows['train']].astype(np.float64).mean(axis=0)
            phi, base = linear_shap(m, X['test'], mean)
            output = base + phi.sum(axis=1)
            logit = m.decision_function(X['test']).astype(np.float64)
            check('linear SHAP: base + sum of phi = logit', np.max(np.abs(output - logit)), 1e-6)
            used, weights, method, space = rows['test'], np.ones(len(rows['test'])), 'linear_shap', 'log-odds'
        elif model == 'xgb':
            import xgboost
            booster, best = m.get_booster(), (0, int(record['training']['best_iteration']) + 1)
            contribs = booster.predict(xgboost.DMatrix(X['test']), pred_contribs=True, iteration_range=best)
            phi, base = contribs[:, :-1].astype(np.float64), float(contribs[0, -1])
            margin = booster.inplace_predict(X['test'], iteration_range=best, predict_type='margin')
            check('TreeSHAP: base + sum of phi = margin', np.max(np.abs(contribs.sum(axis=1) - margin)), 1e-3)
            output = contribs.sum(axis=1)
            used, weights, method, space = rows['test'], np.ones(len(rows['test'])), 'tree_shap', 'log-odds'
        elif seed in settings['rf_shap_seeds']:
            sub = sample_subset(ctx.sample, settings['rf_licit'])
            Xs = X_all[sub['row'].to_numpy()]
            phi, base = tree_shap_forest(m, Xs)
            output = base + phi.sum(axis=1)
            check('TreeSHAP: base + sum of phi = probability', np.max(np.abs(output - score(Xs))), 1e-4)
            used, weights, method, space = sub['row'].to_numpy(), sub['weight'].to_numpy(), 'tree_shap', 'probability'
        else:
            phi = None   # Random Forest on a seed without TreeSHAP (see the timing cell)
        timings['attribution'] = time.perf_counter() - t0
        summary = None
        if phi is not None:
            summary = attribution_summary(phi, values_of(used), data.illicit[used], weights, columns, families, groups)
            matrix = (used, weights, phi, None, base, output, space)
    else:
        import torch
        net = load_network(model, record, model_path)
        kind = 'rows' if model == 'mlp' else 'graph'
        part = {split: period_data(data, X_all, split, kind) for split in ('test', 'val')}
        nodes = {split: (rows[split] if kind == 'rows' else np.flatnonzero(data.split == split)) for split in ('test', 'val')}
        score = {split: network_score(net, part[split].edge_index, part[split].mask) for split in ('test', 'val')}
        test_scores = score['test'](part['test'].x)
        check('reload: largest score difference', np.max(np.abs(test_scores - saved['score'].to_numpy())), 1e-4)
        check('reload: test PR-AUC difference', abs(average_precision_score(y['test'], test_scores) -
                                                    average_precision_score(y['test'], saved['score'].to_numpy())), 1e-4)
        summary = None
        if columns:
            t0 = time.perf_counter()
            perms = {split: within_step_permutations(step[nodes[split]], settings['shuffles']) for split in ('test', 'val')}
            table, restored = permutation_importance(score['test'], part['test'].x, y['test'], units, perms['test'])
            tables.append(table.assign(data='test'))
            if family_units:
                table, _ = permutation_importance(score['val'], part['val'].x, y['val'], family_units, perms['val'])
                tables.append(table.assign(data='val'))
            check('permutation: scores restored', restored, 1e-5)
            timings['permutation'] = time.perf_counter() - t0
            t0 = time.perf_counter()
            if model == 'mlp':
                baseline = torch.tensor(X_all[rows['train']].astype(np.float64).mean(axis=0), dtype=torch.float32,
                                        device=DEVICE)
                phi, base, logits, completeness = integrated_gradients_rows(net, part['test'].x, baseline,
                                                                            settings['ig_steps'])
                check('Integrated Gradients: median completeness error', np.median(completeness), 0.05)
                used, weights, neighbours = rows['test'], np.ones(len(rows['test'])), None
                output = logits
            elif seed in settings['ig_gnn_seeds']:
                train_nodes = np.flatnonzero(data.split == 'train')
                baseline = torch.tensor(X_all[train_nodes].astype(np.float64).mean(axis=0), dtype=torch.float32,
                                        device=DEVICE)
                targets = np.searchsorted(nodes['test'], ctx.sample['row'].to_numpy())
                assert np.array_equal(nodes['test'][targets], ctx.sample['row'].to_numpy())
                ig = gnn_integrated_gradients(net, part['test'], targets, baseline,
                                              receptive_hops(model, record['params']), settings['ig_steps'])
                check('subgraph: largest logit difference from the whole graph', ig.subgraph_difference.max(), 1e-3)
                check('Integrated Gradients: median completeness error', np.median(ig.completeness), 0.05)
                checks.append({'triple': tid, 'check': 'Integrated Gradients: 95th percentile of the completeness error',
                               'value': float(np.percentile(ig.completeness, 95)), 'limit': np.nan, 'ok': True})
                phi, neighbours, output, base = ig.total, ig.neighbours, ig.logits, float('nan')
                used, weights = ctx.sample['row'].to_numpy(), ctx.sample['weight'].to_numpy()
                timings['subgraph_nodes_median'] = float(np.median(ig.sizes))
            else:
                phi = None
            if phi is not None:
                method = 'integrated_gradients'
                summary = attribution_summary(phi, values_of(used), data.illicit[used], weights, columns, families,
                                              groups, neighbours)
                matrix = (used, weights, phi, neighbours, base, output, 'logit')
            timings['attribution'] = time.perf_counter() - t0
        del part
    # ---- files ----
    if tables:
        write_csv(pd.concat(tables, ignore_index=True), dirs.out / 'permutation' / f'{tid}.csv')
    if summary is not None:
        write_csv(summary.assign(method=method, rows=len(matrix[0])), dirs.out / 'shap_summary' / f'{tid}.csv')
        if seed == settings['matrix_seed']:
            used, weights, phi, neighbours, base, output, space = matrix
            table = pd.DataFrame(phi, columns=columns)
            if neighbours is not None:
                table = pd.concat([table, pd.DataFrame(neighbours, columns=[f'{c}__neighbours' for c in columns])], axis=1)
            table.insert(0, 'row', used)
            table.insert(1, 'y', data.illicit[used].astype(int))
            table.insert(2, 'weight', weights)
            table.insert(3, 'base', base)
            table.insert(4, 'output', output)
            atomic(dirs.out / 'shap' / f'{tid}.parquet', lambda tmp: table.to_parquet(tmp, index=False))
    add_checks(dirs.out / 'checks.csv', checks)
    record_out = {'triple': tid, 'menu': menu, 'model': model, 'seed': seed, 'method': method,
                  'attribution_space': matrix[6] if matrix else None, 'timings': timings,
                  'seconds': time.perf_counter() - start, 'settings': {k: v for k, v in settings.items()},
                  'shared_code_version': SHARED_CODE_VERSION,
                  'created': datetime.now(timezone.utc).isoformat(timespec='seconds')}
    write_json(record_out, dirs.out / 'done' / f'{tid}.json')
    free_memory()
    return record_out


def plan_triples(ctx, models, only_pairs=None):
    """Every triple of this notebook, the summaries of step 04, and what is already done."""
    config = ctx.data.config
    pairs = [(menu, model) for menu, model in config['pairs']
             if model in models and (only_pairs is None or f'{menu}__{model}' in only_pairs)]
    triples = [(menu, model, seed) for menu, model in pairs for seed in config['final_runs']['seeds']]
    records = {}
    for menu, model, seed in triples:
        path = ctx.dirs.runs / 'summary' / f'{triple_id(menu, model, seed)}.json'
        assert path.exists(), f'{path} is missing: finish step 04 first'
        r = json.loads(path.read_text(encoding='utf-8'))
        assert r['shared_code_version'] == SHARED_CODE_VERSION, (
            f'{path.name} was written with shared code version {r["shared_code_version"]}, this notebook has '
            f'{SHARED_CODE_VERSION}')
        records[(menu, model, seed)] = r
    ctx.records.update(records)
    expected = {t: expected_work(config, t[0], t[1], ctx.settings) for t in triples}
    done = [t for t in triples if (ctx.dirs.out / 'done' / f'{triple_id(*t)}.json').exists()]
    return triples, Progress(expected, done, ctx.dirs.out / 'logs' / f'06-importance-{NOTEBOOK}.log')


def run_triples(ctx, triples, progress):
    for k, (menu, model, seed) in enumerate(triples, 1):
        if (menu, model, seed) in progress.done:
            continue
        t0 = time.time()
        out = explain_triple(ctx, menu, model, seed)
        progress.finish((menu, model, seed))
        parts = ', '.join(f'{name} {fmt_time(s)}' for name, s in out['timings'].items() if name in ('permutation', 'attribution'))
        progress.print(f'[triple {k}/{len(triples)} · {model} × {menu} · seed {seed}] {parts or "checks only"} · '
                       f'{fmt_time(time.time() - t0)} · {progress.status()}')

In [9]:
# ==== GNNEXPLAINER CODE · GPU notebook only ====
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path


class LogitColumn(nn.Module):
    """The GNN with its logit as an (N, 1) column, the output the explainer of PyG expects."""

    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, x, edge_index):
        return self.net(x, edge_index).view(-1, 1)


def node_classes(data):
    """0 illicit, 1 licit, 2 unknown, for every node."""
    y = data.dataset['y'].to_numpy(dtype=np.float64, na_value=np.nan)
    return np.where(np.isnan(y), 2, np.where(y == 1, 0, 1))


def explain_node(net, part, target, hops, epochs, lr, seed, with_features):
    """GNNExplainer (Ying et al., 2019) for one node: a soft mask in [0, 1] on every edge (and, with features, one mask
    value per column, shared by all nodes), trained to keep the LOGIT of the node (squared error, PyG's 'regression'
    mode) while staying small and close to 0 or 1. Not the class (binary cross-entropy): on a node predicted with
    confidence that loss is saturated, its gradient is about 0, and the masks would stay near their random start.
    Runs on the subgraph within hops + 1 hops, where the logit equals the logit on the whole graph."""
    from torch_geometric.explain import Explainer, GNNExplainer
    from torch_geometric.utils import k_hop_subgraph
    subset, ei, mapping, _ = k_hop_subgraph(int(target), hops + 1, part.edge_index, relabel_nodes=True,
                                            num_nodes=part.x.shape[0])
    set_seed(seed)
    explainer = Explainer(model=LogitColumn(net), algorithm=GNNExplainer(epochs=epochs, lr=lr),
                          explanation_type='model', edge_mask_type='object',
                          node_mask_type='common_attributes' if with_features else None,
                          model_config=dict(mode='regression', task_level='node', return_type='raw'))
    explanation = explainer(part.x[subset], ei, index=int(mapping[0]))
    edge_mask = explanation.edge_mask.detach().double().cpu().numpy()
    feature_mask = explanation.node_mask.detach().double().cpu().numpy().reshape(-1) if with_features else None
    return subset.cpu().numpy(), ei.cpu().numpy(), int(mapping[0]), edge_mask, feature_mask


def edge_geometry(n, ei, centre, hops):
    """Distance of every edge from the explained node (the hop of its farther end), its farther end, and whether its
    message can reach the node: an edge j -> i counts only if i is within hops - 1 of the node (a GNN that looks
    `hops` far passes `hops` messages in a row). The other edges get mask 0 from GNNExplainer by construction
    (no gradient), so they are left out of the means and of the stability check."""
    graph = csr_matrix((np.ones(ei.shape[1]), (ei[0], ei[1])), shape=(n, n))
    dist = shortest_path(graph, directed=False, unweighted=True, indices=centre)
    d_src, d_dst = dist[ei[0]], dist[ei[1]]
    return np.maximum(d_src, d_dst).astype(int), np.where(d_dst >= d_src, ei[1], ei[0]), d_dst < hops


def gnn_strata(record, saved):
    """TP, FN, FP or TN of every test row, at the threshold chosen on validation (step 04)."""
    flagged = saved['score'].to_numpy() >= record['metrics']['thresholds']['best_val_f1']
    y = saved['y'].to_numpy() == 1
    return np.select([flagged & y, ~flagged & y, flagged & ~y], ['TP', 'FN', 'FP'], 'TN')


def explain_pair(ctx, menu, model, gx, progress):
    """GNNExplainer on the core nodes plus `extra_fp` false positives of this model (seed `model_seed`). Files:
    the edges within distance 2 of each node (for the drawings), the mean mask by distance and class of the neighbour
    (every edge whose message can reach the node), the column masks; `<pair>.json` last."""
    data, out = ctx.data, ctx.dirs.out / 'gnnexplainer'
    pair = f'{menu}__{model}'
    if (out / f'{pair}.json').exists():
        return
    start = time.perf_counter()
    seed = gx['model_seed']
    record = ctx.records[(menu, model, seed)]
    net = load_network(model, record, ctx.dirs.runs / record['files']['model'])
    columns = data.config['menus'][menu]['columns']
    fam = column_family(data.config)
    part = period_data(data, feature_matrix(data, menu, 'scaled'), 'test', 'graph')
    nodes = np.flatnonzero(data.split == 'test')
    saved = pd.read_parquet(ctx.dirs.runs / 'predictions' / f'{triple_id(menu, model, seed)}.parquet')
    saved = saved[saved['split'] == 'test']
    strata = dict(zip(saved['row'], gnn_strata(record, saved)))
    core = ctx.sample.loc[ctx.sample['gnnexplainer_core'], 'row'].to_numpy()
    fp = saved.loc[(gnn_strata(record, saved) == 'FP') & ~saved['row'].isin(core), 'row'].to_numpy()
    rng = np.random.default_rng(gx['fp_seed'])
    extra = np.sort(rng.choice(fp, size=min(gx['extra_fp'], len(fp)), replace=False))
    targets = np.concatenate([core, extra])
    hops = receptive_hops(model, record['params'])
    cls = node_classes(data)
    near, by_distance, features = [], [], []
    for k, row in enumerate(targets, 1):
        local = int(np.searchsorted(nodes, row))
        subset, ei, centre, mask, fmask = explain_node(net, part, local, hops, gx['epochs'], gx['lr'],
                                                       gx['explainer_seed'], bool(columns))
        rows_sub = nodes[subset]
        distance, far, reach = edge_geometry(len(subset), ei, centre, hops)
        info = {'target_row': int(row), 'stratum': strata[row], 'core': bool(row in set(core)), 'hops': hops}
        table = pd.DataFrame({'distance': distance, 'neighbour_class': cls[rows_sub[far]], 'mask': mask})[reach]
        agg = table.groupby(['distance', 'neighbour_class'])['mask'].agg(['mean', 'max', 'size']).reset_index()
        by_distance.append(agg.assign(**info))
        keep = distance <= 2
        near.append(pd.DataFrame({'src_row': rows_sub[ei[0][keep]], 'dst_row': rows_sub[ei[1][keep]],
                                  'distance': distance[keep], 'neighbour_class': cls[rows_sub[far[keep]]],
                                  'mask': mask[keep]}).assign(**info))
        if columns:
            features.append(pd.DataFrame({'feature': columns, 'family': [fam[c] for c in columns],
                                          'mask': fmask}).assign(**info))
        if k % 20 == 0 or k == len(targets):
            progress.print(f'[GNNExplainer · {model} × {menu}] {k}/{len(targets)} nodes · '
                           f'{fmt_time(time.perf_counter() - start)}')
    edges = pd.concat(near, ignore_index=True)
    atomic(out / f'{pair}__edges.parquet', lambda tmp: edges.to_parquet(tmp, index=False))
    write_csv(pd.concat(by_distance, ignore_index=True), out / f'{pair}__by_distance.csv')
    if features:
        write_csv(pd.concat(features, ignore_index=True), out / f'{pair}__features.csv')
    write_json({'pair': pair, 'nodes': len(targets), 'core': len(core), 'extra_fp': len(extra), 'hops': hops,
                'settings': gx, 'seconds': time.perf_counter() - start,
                'created': datetime.now(timezone.utc).isoformat(timespec='seconds')}, out / f'{pair}.json')
    del net, part
    free_memory()


def explainer_stability(ctx, menu, model, gx, progress):
    """The edge masks of `stability_nodes` core nodes, explained again: with another seed of the explainer (same
    model), and on the models of the other two seeds (same explainer seed). Agreement with the first explanation:
    Spearman correlation of the masks of the edges within reach of the GNN, and overlap of the 10 strongest edges."""
    data, path = ctx.data, ctx.dirs.out / 'gnnexplainer' / f'{menu}__{model}__stability.csv'
    if path.exists():
        return
    start = time.perf_counter()
    columns = data.config['menus'][menu]['columns']
    part = period_data(data, feature_matrix(data, menu, 'scaled'), 'test', 'graph')
    nodes = np.flatnonzero(data.split == 'test')
    core = np.sort(ctx.sample.loc[ctx.sample['gnnexplainer_core'], 'row'].to_numpy())
    chosen = np.sort(np.random.default_rng(0).choice(core, size=min(gx['stability_nodes'], len(core)), replace=False))
    nets = {s: load_network(model, ctx.records[(menu, model, s)],
                            ctx.dirs.runs / ctx.records[(menu, model, s)]['files']['model'])
            for s in ctx.data.config['final_runs']['seeds']}
    hops = receptive_hops(model, ctx.records[(menu, model, 0)]['params'])
    variants = [('explainer seed 1', 0, 1)] + [(f'model seed {s}', s, 0) for s in nets if s != 0]
    rows = []
    for row in chosen:
        local = int(np.searchsorted(nodes, row))
        subset, ei, centre, reference, _ = explain_node(nets[0], part, local, hops, gx['epochs'], gx['lr'], 0, bool(columns))
        _, _, inside = edge_geometry(len(subset), ei, centre, hops)
        for name, model_seed, explainer_seed in variants:
            _, _, _, mask, _ = explain_node(nets[model_seed], part, local, hops, gx['epochs'], gx['lr'], explainer_seed,
                                            bool(columns))
            a, b = reference[inside], mask[inside]
            k = min(10, len(a))
            rho = spearmanr(a, b).correlation if len(a) > 2 and np.ptp(a) > 0 and np.ptp(b) > 0 else np.nan
            overlap = len(set(np.argsort(-a)[:k]) & set(np.argsort(-b)[:k])) / k if k else np.nan
            rows.append({'pair': f'{menu}__{model}', 'target_row': int(row), 'variant': name, 'edges': int(inside.sum()),
                         'spearman': rho, 'top10_overlap': overlap})
    write_csv(pd.DataFrame(rows), path)
    progress.print(f'[stability · {model} × {menu}] {len(chosen)} nodes x {len(variants)} variants · '
                   f'{fmt_time(time.perf_counter() - start)}')
    del nets, part
    free_memory()

## 2. Load the inputs and the explain sample

`config.json` (step 02) gives menus and families; `04-final-runs/` the models, their summaries and their saved scores. The explain sample is written by the first of the two computing notebooks that runs; the other one rebuilds it and stops if it differs.

In [10]:
DIRS = SimpleNamespace(runs=RUNS_DIR, out=OUT_DIR)
start = time.time()
data = load_inputs(LOCAL_INPUT)
print(f'Inputs loaded in {fmt_time(time.time() - start)}: {len(data.dataset):,} rows, config version {data.config["version"]}.')
assert DEVICE.type == 'cuda', 'No GPU: choose Runtime -> Change runtime type -> T4 GPU, then run again.'
print('GPU:', torch.cuda.get_device_name(0))

if FORCE_RECOMPUTE:
    for menu, model in data.config['pairs']:
        if model in MODELS:
            for folder, suffix in [('done', '.json'), ('permutation', '.csv'), ('shap_summary', '.csv'), ('shap', '.parquet')]:
                for path in (OUT_DIR / folder).glob(f'{menu}__{model}__seed*{suffix}'):
                    path.unlink()
SAMPLE = load_or_build_sample(data, SETTINGS['sample'], OUT_DIR / 'explain_sample.csv')
print(f"Explain sample: {len(SAMPLE):,} rows ({int(SAMPLE['y'].sum())} illicit, {int((SAMPLE['y'] == 0).sum())} licit, "
      f"licit weight {SAMPLE.loc[SAMPLE['y'] == 0, 'weight'].iloc[0]:.3f}), GNNExplainer core {int(SAMPLE['gnnexplainer_core'].sum())}.")
ctx = SimpleNamespace(data=data, dirs=DIRS, settings=SETTINGS, sample=SAMPLE, records={})
TRIPLES, PROGRESS = plan_triples(ctx, MODELS, ONLY_PAIRS)
PROGRESS.print(f'{len(TRIPLES)} triples in this notebook, {len(PROGRESS.done)} already done.')

Inputs loaded in 0.3s: 203,769 rows, config version 1.
GPU: NVIDIA L4
Explain sample: 1,524 rows (524 illicit, 1000 licit, licit weight 9.342), GNNExplainer core 80.
96 triples in this notebook, 96 already done.


## 3. Tests on toy problems

The checks of the CPU notebook (sample, shuffles, permutation importance, summaries), then the networks:
- **Integrated Gradients** of a linear model gives exactly $w_j(x_j - x'_j)$, and on a small MLP, with 200 points, the attributions add up to $f(x) - f(x')$ within 1% (median; on the real models, with 50 points, the error is measured and saved in `checks.csv`);
- a **toy graph of pairs** $u - v$, where the label of $v$ depends only on the feature of its partner $u$, and every $v$ also has two neighbours with random features. GraphSAGE, ChebNet and GATv2 are trained on it, and then: the logit on the subgraph equals the logit on the whole graph; Integrated Gradients gives more than 70% of the attribution to the neighbours; shuffling the column makes the PR-AUC collapse; GNNExplainer gives mask 0 to every edge whose message cannot reach $v$, and puts more mask on the edge **from** the partner into $v$ than on the edges from the distractors (on the 20 positive $v$ with the highest logit of each GNN; at least 40 of the 60 nodes together, by chance 0.4% of the time). This also checks that the edge mask works with each convolution, ChebNet included.

In [11]:
# The explain sample and the shuffles.
counts = largest_remainder(60, [116, 239, 24, 24, 5, 2, 22, 36, 56])      # the illicit of time steps 41-49
assert counts.tolist() == [13, 27, 3, 3, 1, 0, 3, 4, 6], counts
again = build_explain_sample(data, SETTINGS['sample'])
assert np.array_equal(again['row'], SAMPLE['row']) and np.allclose(again['weight'], SAMPLE['weight'])
assert abs(SAMPLE['weight'].sum() - len(eval_rows(data, 'test'))) < 1e-6
core = SAMPLE[SAMPLE['gnnexplainer_core']]
assert (core['y'] == 1).sum() == SETTINGS['sample']['core_illicit'] and (core['y'] == 0).sum() == SETTINGS['sample']['core_licit']
sub = sample_subset(SAMPLE, 200)
assert len(sub) == int(SAMPLE['y'].sum()) + 200 and abs(sub['weight'].sum() - SAMPLE['weight'].sum()) < 1e-6

steps_toy = np.array([1, 1, 1, 2, 2, 3, 3, 3, 3])
values_toy = np.arange(9) * 10
perms_toy = within_step_permutations(steps_toy, 3)
for p in perms_toy:
    assert all(sorted(values_toy[p][steps_toy == t]) == sorted(values_toy[steps_toy == t]) for t in (1, 2, 3))
assert all(np.array_equal(a, b) for a, b in zip(perms_toy, within_step_permutations(steps_toy, 3)))
assert not np.array_equal(perms_toy[0], perms_toy[1])

# Permutation importance on a model with known weights: column 0 decides, column 1 is ignored.
rng = np.random.default_rng(0)
X_toy = rng.standard_normal((2000, 2))
y_toy = (X_toy[:, 0] > 1).astype(int)
score_toy = lambda X: sigmoid64(4 * X[:, 0] + 0 * X[:, 1])
units_toy = {'decisive': ('column', 'A', [0]), 'ignored': ('column', 'A', [1]), 'both': ('family', 'A', [0, 1])}
table_toy, restored = permutation_importance(score_toy, X_toy, y_toy, units_toy, within_step_permutations(np.zeros(2000), 5))
drops = table_toy.set_index('name')['ap_drop']
assert drops['ignored'] == 0 and drops['decisive'] > 0.5 and abs(drops['both'] - drops['decisive']) < 0.05, drops
assert restored == 0

# Summary of attributions: shares add up to 1, the density of a family is its share over its share of columns.
phi_toy = np.column_stack([np.ones(10), 2 * np.ones(10), np.zeros(10)])
s = attribution_summary(phi_toy, phi_toy, np.r_[np.ones(5), np.zeros(5)], np.ones(10), ['a', 'b', 'c'], ['A', 'A', 'B'],
                        {'A': [0, 1], 'B': [2]}).set_index('name')
assert abs(s.loc[['a', 'b', 'c'], 'share'].sum() - 1) < 1e-12 and abs(s.loc['A', 'share'] - 1) < 1e-12
assert abs(s.loc['A', 'density'] - 1.5) < 1e-12 and abs(s.loc['all', 'mean_abs'] - 3) < 1e-12   # floats: no ==
print('Toy tests passed: sample, shuffles within time steps, permutation importance, summaries.')

Toy tests passed: sample, shuffles within time steps, permutation importance, summaries.


In [12]:
# Integrated Gradients of a linear model = the exact formula w_j (x_j - x'_j); completeness on a small MLP.
class LinearToy(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin = nn.Linear(4, 1)

    def forward(self, x, edge_index=None):
        return self.lin(x).squeeze(-1)


torch.manual_seed(0)
linear_net = LinearToy().to(DEVICE)
x_toy = torch.randn(64, 4, device=DEVICE)
base_toy = x_toy.mean(dim=0)
phi, f_base, logits, _ = integrated_gradients_rows(linear_net, x_toy, base_toy, 10)
exact = ((x_toy - base_toy) * linear_net.lin.weight[0]).detach().double().cpu().numpy()
# completeness checked in absolute terms: a relative error explodes on rows where f(x) - f(x') is close to 0
assert np.allclose(phi, exact, atol=1e-5) and np.allclose(phi.sum(axis=1), logits - f_base, atol=1e-5)
mlp_toy = MLP(4, 16, 2, 0.0).to(DEVICE)
_, _, _, err = integrated_gradients_rows(mlp_toy, x_toy, base_toy, 200)   # the error of the rule shrinks with more points
assert np.median(err) < 0.01, np.median(err)


# A toy graph of pairs u - v, where the label of v depends only on the feature f of its partner u; every v also has
# two distractor neighbours with random f. Column 0 = f, column 1 = role (1 for the u nodes). A GNN that learned the
# task must: give the same logit on the subgraph as on the whole graph; give most of the attribution to the
# neighbours; lose its PR-AUC when f is shuffled; and, with GNNExplainer, put more mask on the edge from the partner
# than on the edges from the distractors.
def toy_pairs(seed, n=400):
    rng = np.random.default_rng(seed)
    f = rng.standard_normal(3 * n)                         # u: 0..n-1, v: n..2n-1, distractors: 2n..3n-1
    role = np.r_[np.ones(n), np.zeros(2 * n)]
    y = np.zeros(3 * n)
    y[n:2 * n] = (f[:n] > 0.8)
    src = np.r_[np.arange(n), rng.integers(2 * n, 3 * n, size=2 * n)]
    dst = np.r_[np.arange(n) + n, np.repeat(np.arange(n) + n, 2)]
    mask = np.zeros(3 * n, dtype=bool)
    mask[n:2 * n] = True
    edge_index = to_undirected(torch.tensor(np.vstack([src, dst]), dtype=torch.long), num_nodes=3 * n).to(DEVICE)
    return SimpleNamespace(x=torch.tensor(np.column_stack([f, role]), dtype=torch.float32, device=DEVICE), edge_index=edge_index,
                           y=torch.tensor(y, dtype=torch.float32, device=DEVICE), mask=torch.tensor(mask, device=DEVICE),
                           y_eval=y[mask].astype(int), partner=np.arange(n))


toy_train, toy_val = toy_pairs(0), toy_pairs(1)
gx_wins = {}
toy_weight = float((toy_train.y_eval == 0).sum() / toy_train.y_eval.sum())
for model, params in [('sage', {'layers': 2, 'hidden': 32, 'aggr': 'mean', 'variant': 'lin', 'dropout': 0.0}),
                      ('cheb', {'layers': 2, 'hidden': 32, 'K': 2, 'variant': 'lin', 'dropout': 0.0}),
                      ('gatv2', {'layers': 2, 'heads': 2, 'hidden_per_head': 16, 'att_dropout': 0.0, 'variant': 'lin',
                                 'dropout': 0.0})]:
    set_seed(0)
    net = build_neural(model, params, 2, {'normalization': 'sym'} if model == 'cheb' else {})
    result = train_neural(net, toy_train, toy_val, 1e-2, 0.0, toy_weight, 300, 60, keep_best_state=True)
    net.load_state_dict(result['best_state'])
    net.eval()
    assert result['best_ap'] > 0.8, (model, result['best_ap'])
    hops = receptive_hops(model, params)
    targets = np.flatnonzero(toy_val.mask.cpu().numpy())[:40]
    ig = gnn_integrated_gradients(net, toy_val, targets, toy_train.x.mean(dim=0), hops, 50)
    assert ig.subgraph_difference.max() < 1e-4, (model, ig.subgraph_difference.max())
    assert np.median(ig.completeness) < 0.01, (model, np.median(ig.completeness))
    nb_share = np.abs(ig.neighbours).sum() / (np.abs(ig.total - ig.neighbours).sum() + np.abs(ig.neighbours).sum())
    assert nb_share > 0.7, (model, nb_share)
    score = network_score(net, toy_val.edge_index, toy_val.mask)
    table, restored = permutation_importance(score, toy_val.x.clone(), toy_val.y_eval, {'f': ('column', 'A', [0])},
                                             within_step_permutations(np.zeros(toy_val.x.shape[0]), 3))
    assert table['ap_drop'][0] > 0.3 and restored < 1e-6, (model, table['ap_drop'][0])
    # GNNExplainer, with the settings of the real run, on the 20 positive v nodes with the highest logit (there the
    # model is sure because of the partner). For every node and every GNN: edges whose message cannot reach v get
    # mask 0, some edge within reach gets more (the subgraph, the explained node and the order of the edges line up,
    # and the mask acts on this convolution). The signal: the messages that reach v at the last step come along the
    # edges INTO v (the edge v -> u carries only the noise of v), and the edge from the partner should get more mask
    # than those from the distractors. How often depends on how each toy model learned (a GraphSAGE with mean
    # aggregation also leans on its distractors), so the check is on the three GNNs together, after the loop.
    with torch.no_grad():
        toy_logit = net(toy_val.x, toy_val.edge_index).cpu().numpy()
    positives = np.flatnonzero(toy_val.y.cpu().numpy() == 1)
    positives, wins = positives[np.argsort(-toy_logit[positives], kind='stable')[:20]], 0
    for t in positives:
        subset, ei, centre, mask, _ = explain_node(net, toy_val, int(t), hops, GNNEXPLAINER['epochs'], GNNEXPLAINER['lr'],
                                                   0, True)
        _, _, reach = edge_geometry(len(subset), ei, centre, hops)
        assert (mask[~reach] == 0).all() and (mask[reach] > 0).any(), (model, 'edge masks out of place')
        into = ei[1] == centre
        from_partner = into & (subset[ei[0]] == int(t) - 400)
        wins += mask[from_partner].mean() > mask[into & ~from_partner].mean()
    gx_wins[model] = int(wins)
    print(f'{model}: toy PR-AUC {result["best_ap"]:.2f}; subgraph = whole graph; IG completeness ok, '
          f'neighbour share {nb_share:.2f}; shuffling the column: PR-AUC -{table["ap_drop"][0]:.2f}; '
          f'GNNExplainer prefers the edge from the partner on {wins}/20 nodes')
# 40 of 60: random masks get there 0.4% of the time (locally, on stand-ins faithful to PyG: 51 to 60 of 60)
assert sum(gx_wins.values()) >= 40, gx_wins
print('Toy tests passed: Integrated Gradients, subgraphs, permutation and GNNExplainer on the networks.')

sage: toy PR-AUC 0.98; subgraph = whole graph; IG completeness ok, neighbour share 0.73; shuffling the column: PR-AUC -0.79; GNNExplainer prefers the edge from the partner on 15/20 nodes
cheb: toy PR-AUC 0.99; subgraph = whole graph; IG completeness ok, neighbour share 0.81; shuffling the column: PR-AUC -0.81; GNNExplainer prefers the edge from the partner on 20/20 nodes
gatv2: toy PR-AUC 1.00; subgraph = whole graph; IG completeness ok, neighbour share 0.82; shuffling the column: PR-AUC -0.81; GNNExplainer prefers the edge from the partner on 20/20 nodes
Toy tests passed: Integrated Gradients, subgraphs, permutation and GNNExplainer on the networks.


## 4. Timing

Measured on the slowest GNN pair of this notebook (largest receptive field times number of columns), seed 0: one forward pass on the test graph, Integrated Gradients on 5 nodes of the sample, GNNExplainer on 2 nodes. From these, the estimate of the time of each part, and the seeds of Integrated Gradients on the GNNs (`'auto'`: all three if the estimate fits in `ig_budget_hours`, otherwise seed 0 only, as decided in F8). The choice is saved in `settings-gpu.json` the first time, so that every session uses the same one.

In [13]:
gnn_pairs = [(m, mo) for m, mo in dict.fromkeys((t[0], t[1]) for t in TRIPLES)
             if mo in GNN_MODELS and data.config['menus'][m]['columns']]
if gnn_pairs:
    cost = lambda p: receptive_hops(p[1], ctx.records[(p[0], p[1], 0)]['params']) * (len(data.config['menus'][p[0]]['columns']) + 1)
    menu, model = max(gnn_pairs, key=cost)
    record = ctx.records[(menu, model, 0)]
    net = load_network(model, record, RUNS_DIR / record['files']['model'])
    X_all = feature_matrix(data, menu, 'scaled')
    part = period_data(data, X_all, 'test', 'graph')
    nodes = np.flatnonzero(data.split == 'test')
    score = network_score(net, part.edge_index, part.mask)
    score(part.x)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(5):
        score(part.x)
    t_forward = (time.perf_counter() - t0) / 5
    targets = np.searchsorted(nodes, SAMPLE['row'].to_numpy()[:5])
    baseline = torch.tensor(X_all[data.split == 'train'].mean(axis=0), dtype=torch.float32, device=DEVICE)
    t0 = time.perf_counter()
    gnn_integrated_gradients(net, part, targets, baseline, receptive_hops(model, record['params']), SETTINGS['ig_steps'])
    t_ig = (time.perf_counter() - t0) / 5
    t0 = time.perf_counter()
    for t in targets[:2]:
        explain_node(net, part, int(t), receptive_hops(model, record['params']), GNNEXPLAINER['epochs'], GNNEXPLAINER['lr'], 0, True)
    t_gx = (time.perf_counter() - t0) / 2
    del net, part
    free_memory()
    n_pairs = len(gnn_pairs)
    ig_all = n_pairs * 3 * len(SAMPLE) * t_ig
    settings_file = OUT_DIR / 'settings-gpu.json'
    if settings_file.exists():
        SETTINGS['ig_gnn_seeds'] = json.loads(settings_file.read_text(encoding='utf-8'))['ig_gnn_seeds']
        print(f'Choice of an earlier session ({settings_file.name}) kept.')
    else:
        if SETTINGS['ig_gnn_seeds'] == 'auto':
            SETTINGS['ig_gnn_seeds'] = [0, 1, 2] if ig_all <= SETTINGS['ig_budget_hours'] * 3600 else [0]
        write_json({'ig_gnn_seeds': SETTINGS['ig_gnn_seeds'], 'seconds_per_node': t_ig, 'forward_seconds': t_forward,
                    'gnnexplainer_seconds_per_node': t_gx}, settings_file)
    units = sum(len(menu_units(data.config, m)[0]) + len(menu_units(data.config, m, True)[0]) for m, mo in gnn_pairs) * 3
    print(f'Slowest pair {model} x {menu}: forward pass {t_forward * 1000:.0f} ms, Integrated Gradients {t_ig:.2f}s per node, '
          f'GNNExplainer {t_gx:.1f}s per node.')
    print(f"Upper estimates (every pair as slow as this one): permutation of the GNNs ~{fmt_time(units * SETTINGS['shuffles'] * t_forward)}, "
          f"Integrated Gradients ~{fmt_time(n_pairs * len(SETTINGS['ig_gnn_seeds']) * len(SAMPLE) * t_ig)} "
          f"(seeds {SETTINGS['ig_gnn_seeds']}), GNNExplainer ~{fmt_time((n_pairs + 2) * (100 + 80) * t_gx)}.")
else:
    SETTINGS['ig_gnn_seeds'] = [0, 1, 2] if SETTINGS['ig_gnn_seeds'] == 'auto' else SETTINGS['ig_gnn_seeds']
for key in PROGRESS.expected:
    PROGRESS.expected[key] = expected_work(data.config, key[0], key[1], SETTINGS)

Choice of an earlier session (settings-gpu.json) kept.
Slowest pair cheb x local_agg_graph: forward pass 17 ms, Integrated Gradients 0.15s per node, GNNExplainer 3.8s per node.
Upper estimates (every pair as slow as this one): permutation of the GNNs ~6m 49s, Integrated Gradients ~1h 09m (seeds [0]), GNNExplainer ~3h 45m.


## 5. Permutation importance and attributions

One log line per triple, for example

```
[triple 40/96 · sage × transparent · seed 0] permutation 1m 10s, attribution 4m 30s · 5m 45s · done 40, left 56 · elapsed 1h 20m, left ~2h 10m
```

Before explaining a triple, the reloaded model must give back the saved scores of step 04 (exactly for XGBoost up to $10^{-6}$; for the networks within $10^{-4}$, the rounding of the GPU), otherwise the notebook stops. For the GNNs, the logit of every explained node on its subgraph must equal its logit on the whole graph.

In [14]:
run_triples(ctx, TRIPLES, PROGRESS)

## 6. GNNExplainer and its stability

For every (menu, GNN) pair: the 80 core nodes plus 20 false positives of the model (seed 0), then the stability check on 20 core nodes. One line every 20 nodes.

In [15]:
gx_pairs = [(m, mo) for m, mo in dict.fromkeys((t[0], t[1]) for t in TRIPLES) if mo in GNN_MODELS]
for k, (menu, model) in enumerate(gx_pairs, 1):
    explain_pair(ctx, menu, model, GNNEXPLAINER, PROGRESS)
    explainer_stability(ctx, menu, model, GNNEXPLAINER, PROGRESS)
    PROGRESS.print(f'[GNNExplainer] pair {k}/{len(gx_pairs)} done: {model} × {menu}')

[GNNExplainer] pair 1/20 done: sage × local
[GNNExplainer] pair 2/20 done: sage × local_agg
[GNNExplainer] pair 3/20 done: sage × graph
[GNNExplainer] pair 4/20 done: sage × econ
[GNNExplainer] pair 5/20 done: sage × transparent
[GNNExplainer] pair 6/20 done: sage × local_agg_graph
[GNNExplainer] pair 7/20 done: gatv2 × local
[GNNExplainer] pair 8/20 done: gatv2 × local_agg
[GNNExplainer] pair 9/20 done: gatv2 × graph
[GNNExplainer] pair 10/20 done: gatv2 × econ
[GNNExplainer] pair 11/20 done: gatv2 × transparent
[GNNExplainer] pair 12/20 done: gatv2 × local_agg_graph
[GNNExplainer] pair 13/20 done: cheb × local
[GNNExplainer] pair 14/20 done: cheb × local_agg
[GNNExplainer] pair 15/20 done: cheb × graph
[GNNExplainer] pair 16/20 done: cheb × econ
[GNNExplainer] pair 17/20 done: cheb × transparent
[GNNExplainer] pair 18/20 done: cheb × local_agg_graph
[GNNExplainer] pair 19/20 done: cheb × none
[GNNExplainer] pair 20/20 done: gin × none


## 7. Summary

Triples and pairs finished, and the checks of this notebook (every row of `checks.csv` has a limit; the notebook would have stopped at the first failure).

In [16]:
mine = [triple_id(*t) for t in TRIPLES]
done = [t for t in mine if (OUT_DIR / 'done' / f'{t}.json').exists()]
gx_done = [p for p in gx_pairs if (OUT_DIR / 'gnnexplainer' / f'{p[0]}__{p[1]}.json').exists()]
print(f'Triples of this notebook finished: {len(done)} of {len(mine)}; GNNExplainer pairs: {len(gx_done)} of {len(gx_pairs)}.')
assert len(done) == len(mine) and len(gx_done) == len(gx_pairs), 'something is missing: run the notebook again'
checks = pd.read_csv(OUT_DIR / 'checks.csv')
checks = checks[checks['triple'].isin(mine)]
display(checks.groupby('check').agg(triples=('triple', 'count'), largest=('value', 'max'), limit=('limit', 'first'),
                                    all_ok=('ok', 'all')))
drive.flush_and_unmount()
print('Every file is on Drive in', OUT_DIR)

Triples of this notebook finished: 96 of 96; GNNExplainer pairs: 20 of 20.


,triples,largest,limit,all_ok
check,,,,
Integrated Gradients: 95th percentile of the completeness error,18,5.294282e-01,NaN,True
Integrated Gradients: median completeness error,36,3.868422e-02,5.000000e-02,True
TreeSHAP: base + sum of phi = margin,18,1.430511e-05,1.000000e-03,True
permutation: scores restored,90,0.000000e+00,1.000000e-09,True
reload: largest score difference,96,6.309925e-07,1.000000e-06,True
reload: test PR-AUC difference,78,3.744038e-06,1.000000e-04,True
subgraph: largest logit difference from the whole graph,18,7.247925e-05,1.000000e-03,True


Every file is on Drive in /content/drive/MyDrive/UNIVERSITA/block-crypto/data/06-feature-importance
